In [ ]:
import numpy as np
from scipy import stats
import pandas as pd
from itertools import combinations
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
%matplotlib inline

### Data 145 Spring 2026
#### A. Adhikari and W. Fithian

# Worksheet 8 Part 2 #

## 4. Comparing Two Small Samples

A pharmaceutical lab has two methods (named A and B) for measuring the amount of a particular chemical in tablets. It uses both methods on each of 9 tablets. The data, measured in milligrams, are given below. Each row corresponds to the pair of measurements on a single tablet. You can assume that the pairs of measurements on different tablets are i.i.d. However, the lab is not willing to assume that the two methods applied to the same tablet are independent, and it is not sure whether they are identically distributed.

In [ ]:
method_A = np.array([93.7, 102.3, 96.0, 96.5, 91.7, 87.3, 92.7, 92.8, 87.5])
method_B = np.array([97.2, 97.8, 96.2, 101.8, 94.8, 95.8, 98.0, 99.0, 100.2])
data_dict = {'A': method_A, 'B': method_B}
data = pd.DataFrame(data_dict)

In [ ]:
data

**a) Parametric, Paired:** Assume that each pair is bivariate normal, and perform a parametric test of whether or not the two methods agree on average. Your answer should include:
1. The precise hypotheses you are testing. It's a good idea to set up some notation.
2. The test statistic you will use, and its numerical value.
3. The null distribution of the test statistic.
4. The $p$-value of the test.
5. Whether the test rejects the null hypothesis at the 5\% level, and whether it rejects at the 1\% level.

Useful code: 
- `np.std(x, ddof=1)` evaluates to $\frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$ for the numerical array `x`.
- `stats.norm.cdf(z)` evaluates to the standard normal cdf at the point `z`, and `stats.t.cdf(w, df=k)` evaluates to the cdf of the $t$ distribution with $k$ degrees of freedom, evaluated at the point `w`.

In [ ]:
# Answer to Part (a)

... # Use as many lines as you need
statistic, p_value # evaluates to test statistic, p-value

**b) Parametric, Independent Samples:** Here is a scatter plot of the data. Answer the following question ***based on the scatter plot alone,*** without doing any numerical calculations.

If you had treated the samples from the two methods as independent of each other, and tested whether the two methods agree on average, would the value of your test statistic have been closer to 0 than the statistic in Part (a) or would it have been further away? Why?

In [ ]:
plt.scatter(method_A, method_B);

**c) Non-parametric, paired:** If you perform the sign test of whether or not the joint distribution of $(X, Y)$ is symmetric, what will the value of the statistic be? Answer by simply reading it off the `data` table at the start of the question.

Then complete the cell below to perform the test. In the blanks in the comments at the end, enter Yes or No.

In [ ]:
# Answer to Part (c)

sign_test_statistic =  ...
sign_test_p_value = ...
sign_test_statistic, sign_test_p_value

# Reject the null at the 5% level? ...
# Reject the null at the 1% level? ...

**d) Effect of Small Sample:** In Part (c), what percent of positive signs would you expect under the null hypothesis? What percent did you observe? How do you reconcile the difference between those two values with your conclusions in Part (c)? 

## 5. A Rank-Based Test
We will develop the method by revisiting Deflategate, a storm in the world of American football and familiar to many of us from Data 8.

Here are some extracts from the Data 8 textbook:

"On January 18, 2015, the Indianapolis Colts and the New England Patriots played the American Football Conference (AFC) championship game to determine which of those teams would play in the Super Bowl. After the game, there were allegations that the Patriots' footballs had not been inflated as much as the regulations required; they were softer. This could be an advantage, as softer balls might be easier to catch ...

At half-time, all the game balls were collected for inspection. Two officials, Clete Blakeman and Dyrol Prioleau, measured the pressure in each of the balls. 

Here are the data. Each row corresponds to one football. Pressure is measured in psi [pounds per square inch]. The Patriots ball that had been intercepted by the Colts was not inspected at half-time. Nor were most of the Colts' balls – the officials simply ran out of time and had to relinquish the balls for the start of second half play."

Each team had 12 footballs. Eleven of the Patriots' footballs were measured, and four of the Colts'. They were measuring at half-time, and they ran out of time. The teams had to go back and play.

In [ ]:
footballs = pd.read_csv('deflategate.csv')
footballs

It is clear that the Patriots' footballs had less pressure than the Colts'. But that is not a fair comparison since the two sets of footballs started out at different pressures: all the Patriots' footballs at 12.5 psi and the Colts' at 13 psi, both levels allowed by NFL regulations. 

Pressure drops naturally during the game. The variable of interest, therefore, is the amount by which the pressure dropped. The Colts' allegation can be politely restated as saying that the drops in pressure among the Patriots' footballs were so large that something unusual had to have happened.

Let's work with drops in pressure instead.

In [ ]:
starts = np.append(12.5*np.ones(11), 13*np.ones(4))
blakeman_drops = starts - footballs['Blakeman'].values
prioleau_drops = starts - footballs['Prioleau'].values
drops = pd.DataFrame({'Blakeman': blakeman_drops, 'Prioleau': prioleau_drops})
drops

### Ranks in Python
It does look as though the pressure drop among the Colts' footballs was less than that among the Patriots'. To examine this further, we have to first figure out how to deal with the fact that the two officials' measurements were quite different from each other. 

In Data 8 we averaged the two drops for each football and used those 15 averages as our data. We then used permutations to test for the equality of the underlying means. 

Now let's look at each official's data separately. Since we are just interested in the ordering of the drops and not their values, we will see how the two officials' rankings compare.

The `stats` function `rankdata` takes a numerical array as its argument and returns the array of ranks.

In [ ]:
data_1 = np.array([27, 32, 28, 35, 25])
stats.rankdata(data_1)

When we use rank-based methods we do have to face the issue of "ties," that is, data values that are equal. For what we are going to do in this question, it doesn't matter how you rank tied values. We ask that you rank ties by using the `method = 'ordinal'` option of `rankdata`. It assigns distinct ranks to all the values, assigning consecutive ranks to equal values in the order in which they appear in the data.

In [ ]:
data_2 = np.append(data_1, 32 * np.ones(3))
data_2, stats.rankdata(data_2, method = 'ordinal')

**a) Ranking the two officials' samples:** Use `rankdata` with the `method = 'ordinal'` option to rank Blakeman's drop values, and, separately, Prioleau's drop values. Augment `drops` with two columns labeled `Blakeman Rank` and `Prioleau Rank`. Use as many lines as you need. 

Make sure to look at the ranks and check for consistency.

In [ ]:
#Answer to Part (a)

blakeman_ranks = ...
prioleau_ranks = ...
...
drops

**b)** In which columns is it easier to compare consistency and inconstency between the two officials: the ranks or the drop values? What consistencies and inconsistencies do you notice when you compare the ranks?

**Your answer here.**

### Performing the Test
Is the difference due to chance? More precisely, the question is whether the Colts' ranks are like a simple random sample from all 15 ranks or whether the Colts' ranks are generally smaller than the Patriots'. If they are smaller, then it means that by comparison, the pressure in the Patriots' footballs dropped by more than can't be explained by random chance. That is what the Colts were alleging. 

In fact, the Colts were alleging even more, which is that the increased drop was deliberate. We can't assess that. But we can see whether the the Colts' ranks are generally too low to be explained by chance.

It is now time to quantify "ranks are generally too low". We will do this by using the Wilcoxon Rank Sum statistic, which is just the sum of the Colts' ranks. 

It is important to keep in mind that we are not interested in which of the Colts' footballs received which rank. We are just interested in the set of ranks received by those balls. That is, we are interested in an unordered sample of 4 out of the 15 ranks.

**d) Wilcoxon's Rank Sum Statistic:** What is the rank sum statistic based on Blakeman's ranks? That is, what is the sum of the Colts' ranks as assigned by Blakeman?

What is your informal conclusion about whether Blakeman's ranks look like a random sample of 4 out of the 15 ranks? Explain.

**It's important to keep in mind that your conclusion doesn't identify a reason for what you have observed.** In particular, it doesn't imply that anything illegal happened.

**Your answer here.**

**e)** The dataframe `all_samples` consists of just one column containing all possible subsets of 4 ranks out of 15.

In [ ]:
population = np.arange(1, 16)
all_samples = pd.DataFrame({'Rank Set': list(combinations(population, 4))})
all_samples

Create an array `rank_sums` of the sums of the ranks of all the subsets. Augment the dataframe with a column labeled `Rank Sum` that contains the sum of the ranks in each set.

In [ ]:
# Answer to Part (e)

# Use as many lines as you need.
...
rank_sums = ... 
... 
all_samples

**f) Null distribution of Wilcoxon's Rank Sum statistic:** Under the null hypothesis of 4 random draws from the ranks 1 through 15, all rows of `all_samples` are equally likely. Enter the lowest and highest rank sum to draw all bars of the histogram of the null distribution of the statistic. 

It should be a familiar shape. Note that this is not an approximation. It's an exact distribution calculated by complete enumeration.

In [ ]:
# Answer to Part (e)

lowest_sum = ...
highest_sum = ...
plt.hist(rank_sums, bins=np.arange(lowest_sum-0.5, highest_sum+0.6, 1), color='tab:blue', edgecolor='w', density=True);

**g) Expectation and SD of the null distribution:** Complete each blank in the cell below with a single expression, to find the expectation and standard deviation of the Wilcoxon rank sum statistic W under the null hypothesis of random draws. **Do not** use `np.mean` and `np.std`. Use the formulas given in class, and do not use additional lines of code.

Check that the numbers look correct on your histogram. Then run the final cell and confirm that your answers are correct.

In [ ]:
#Answer to Part (g)

# Expectation
e_W = ...

# Standard deviation of W
sd_W = ...

e_W, sd_W

In [ ]:
np.mean(rank_sums), np.std(rank_sums)

**h) Prioleau's ranks:** Do Prioleau's ranks of the Colts data look like a random sample from all ranks? Find the one-tailed $p$-value and state your conclusion at level 5\%. You shouldn't have to do a lot of work. 

In [ ]:
# Answer to Part (h)

prioleau_statistic = ...
prioleau_p_value = ...
prioleau_p_value # Reject H_0 at level 5%? Answer: ...